# 数据结构与科学计算 · 第 03 讲
## 哈希表、字典、集合与 CFD 数据索引

> **课程定位**：面向已经学习 Python 基础、正在准备自行编写二维 CFD 求解器的学习者。本讲从数据结构原理出发，以实际数值计算问题作为应用练习。

**完成本讲后，你应该能够：**

1. 理解哈希函数、哈希冲突与平均 $O(1)$ 查找的含义；
2. 正确使用 `dict` / `set`，区分可哈希对象和可变对象；
3. 根据稠密或稀疏的访问模式，选择数组或哈希表；
4. 使用键值对表示网格边界标签、缓存重复数值计算；
5. 认识浮点数键、数据一致性和 CPU/GPU 存储的边界。

**预计学习时间**：80–120 分钟。所有核心示例可在 VS Code 的 Jupyter Notebook 中直接运行。

## 0. 使用方法与运行环境

1. 在 VS Code 打开此 `.ipynb`，右上角选择当前项目的 Python Kernel；
2. 使用 `Shift + Enter` 依次运行。修改代码后，建议 **Restart Kernel & Run All** 以消除旧变量干扰；
3. **练习单元以 `TODO` 留空**，后面会有逐项自动检查；未完成练习时显示“尚未通过”是预期行为；
4. 最后附有独立命名的 `_solution` 参考实现，不会覆盖你的练习函数；
5. 学习重点是“选择什么结构、为什么、代价是什么”，不只是背诵 API。

前两讲已安装 NumPy。本系列推荐运行 `uv pip install numpy scipy matplotlib ipykernel jupyter`。第 08 讲还需要 JAX：`uv pip install -U jax`（此命令默认安装适用于当前平台的发行版，GPU 支持视平台而定）。

In [1]:
import sys
print('Python:', sys.version.split()[0])
print('Notebook 已连接到 Python Kernel')

Python: 3.13.5
Notebook 已连接到 Python Kernel


## 1. 从问题出发：为什么 `in` 有时很慢？

假设一个二维网格里有一批需要特殊处理的网格单元，我们要反复判断坐标 `(j,i)` 是否属于“固体区”。

- 用 `list` 保存坐标：需要逐个比较，最坏 $O(n)$；
- 用 `set` 保存坐标：通常只需计算哈希并定位存储位置，平均 $O(1)$；
- 用布尔 NumPy 掩码：如果坐标在固定致密网格上，则是直接索引，而且更便于向量化。

这里说的 **平均/摊销** 并非保证所有情况都为 $O(1)$。哈希冲突、扩容和对抗性输入都可能影响复杂度。

In [2]:
import time
import numpy as np

N = 40_000
coordinates = [(k // 200, k % 200) for k in range(N)]
coordinate_set = set(coordinates)
query = (199, 199)

for name, container in [('list', coordinates), ('set', coordinate_set)]:
    t0 = time.perf_counter()
    for _ in range(100):
        found = query in container
    dt = (time.perf_counter()-t0)*1e3
    print(f'{name:>4}: found={found}, 100次查询约 {dt:.2f} ms')

mask = np.zeros((200,200), dtype=bool)
mask[199,199] = True
print('dense mask:', bool(mask[query]))

list: found=True, 100次查询约 52.04 ms
 set: found=True, 100次查询约 0.01 ms
dense mask: True


## 2. 哈希表的工作原理

可以把字典想象为一个按键定位的表：

1. 对键 $k$ 计算整数哈希 `hash(k)`；
2. 将哈希信息映射到内部存储位置；
3. 若发生冲突，使用实现内部的冲突处理机制继续查找；
4. 比较候选键是否相等，最终找到对应的值。

Python 的 `dict` 和 `set` 是哈希表的高层接口。**不要把 `hash(key) % len(dict)` 当作 CPython 的真实内部实现**，它只是帮助理解的示意。

复杂度：

| 操作 | `dict` / `set` 典型平均复杂度 | 备注 |
|:--|:--:|:--|
| 插入 / 查询 / 删除 | $O(1)$ | 冲突和扩容可能改变某次操作的耗时 |
| 遍历全部元素 | $O(n)$ | 与容器大小成正比 |
| 集合交集 | 取决于两集合规模 | 通常远快于手写双层循环 |

`dict` 保留**插入顺序**（语言保证），但不代表键按数值大小排序。

In [3]:
student = {'name': 'grid_solver', 'nx': 128, 'ny': 64}
student['dtype'] = 'float64'
print('keys:', list(student))
print('nx=',student.get('nx'), 'nz with default=',student.get('nz', 1))

visits = [(0, 0), (0, 1), (0, 0), (1, 0)]
unique = set(visits)
print('visited cells:', unique)
print('set intersection:', {1,2,3} & {2,3,4})

keys: ['name', 'nx', 'ny', 'dtype']
nx= 128 nz with default= 1
visited cells: {(0, 1), (1, 0), (0, 0)}
set intersection: {2, 3}


### 2.1 为什么列表不能作为 `dict` 的键？

键必须是 **可哈希的**，且在作为键期间其哈希值以及相等性不能发生影响字典一致性的变化。整数、字符串和仅包含可哈希成员的元组通常可哈希；列表和普通字典不可哈希。元组包含列表时也不可哈希。

注意：**可哈希并不严格等同于完全不可变**；自定义对象可以实现 `__hash__`，但必须遵守 `a == b` 时 `hash(a) == hash(b)` 的约束。

In [4]:
print('tuple hash:', hash((3, 7)))
for x in ([3, 7], (3, [7])):
    try:
        print('hash =', hash(x))
    except TypeError as exc:
        print('不可哈希:',type(x).__name__,str(exc))

# 不应使用 NaN 作通用网格索引键，也尽量避免未离散化的浮点坐标作为键
print('0.1 + 0.2 == 0.3 ?', 0.1 + 0.2 == 0.3)

tuple hash: 4803095124130366729
不可哈希: list unhashable type: 'list'
不可哈希: tuple unhashable type: 'list'
0.1 + 0.2 == 0.3 ? False


## 3. 三种“二维数据”的存储方案

假设 CFD 计算域大小为 $n_y\times n_x$：

| 存储 | 适用情况 | 代价 |
|:--|:--|:--|
| `ndarray[ny,nx]` | **所有单元**都有值、批量计算 | 占用 $O(n_xn_y)$ 个数值槽位 |
| `set[(j,i)]` | 仅保存**少量特殊单元的坐标** | 有哈希和 Python 对象开销 |
| `dict[(j,i)] -> value` | 非均匀边界标签、稀疏元数据 | 随存储项数增长的 Python 对象开销 |

一个非常重要的区别：**“稀疏元数据”可以用字典，但稀疏线性代数矩阵应优先用 SciPy CSR/COO**（第 07 讲）。

In [5]:
ny, nx = 7, 9
field = np.zeros((ny, nx))
wall_cells = {(j, 0) for j in range(ny)} | {(j, nx-1) for j in range(ny)}
wall_mask = np.zeros((ny,nx), dtype=bool)
for j, i in wall_cells:
    wall_mask[j,i] = True

bc_tags = {(j, 0): 'inlet' for j in range(ny)}
bc_tags.update({(j, nx-1): 'outlet' for j in range(ny)})
print('固体/特殊边界数:', len(wall_cells))
print('某边界标签:',bc_tags[(2,0)],bc_tags[(2,nx-1)])
print(wall_mask.astype(int))

固体/特殊边界数: 14
某边界标签: inlet outlet
[[1 0 0 0 0 0 0 0 1]
 [1 0 0 0 0 0 0 0 1]
 [1 0 0 0 0 0 0 0 1]
 [1 0 0 0 0 0 0 0 1]
 [1 0 0 0 0 0 0 0 1]
 [1 0 0 0 0 0 0 0 1]
 [1 0 0 0 0 0 0 0 1]]


### 3.1 组合使用 `dict` + `set`

`set` 适合筛选节点；`dict` 适合映射节点到物理量、标签或历史记录。下面尝试统计连通单元的边界类型：

In [6]:
from collections import Counter
print('boundary count:', Counter(bc_tags.values()))
for cell, tag in list(bc_tags.items())[:3]:
    print('cell',cell,'type',tag)

# 简洁的坐标规范化：离散下标使用整数，而不是把近似浮点数直接作字典键
def cell_from_point(x, y, dx, dy):
    return (int(np.floor(y/dy)), int(np.floor(x/dx)))
print('位置映射单元:', cell_from_point(0.31,0.56,0.1,0.1))

boundary count: Counter({'inlet': 7, 'outlet': 7})
cell (0, 0) type inlet
cell (1, 0) type inlet
cell (2, 0) type inlet
位置映射单元: (5, 3)


## 4. 哈希缓存（Memoization）：什么时候有用？

缓存适合输入重复、计算昂贵、函数结果确定的情况。Python 提供 `functools.lru_cache`。如果数值数组不断变化，不可直接把整个可变 `ndarray` 作为缓存键。

**易踩的坑**：缓存用于纯函数（相同输入意味着相同输出），否则会复用已过时的结果。

In [7]:
from functools import lru_cache

@lru_cache(maxsize=1024)
def stencil_offsets(radius: int):
    # 构造 L1 距离不超过 radius 的邻域偏移
    return tuple((dj, di)
                 for dj in range(-radius, radius+1)
                 for di in range(-radius, radius+1)
                 if abs(dj) + abs(di) <= radius)

print('radius 2:',stencil_offsets(2))
print('cache before:',stencil_offsets.cache_info())
for _ in range(10):
    stencil_offsets(2)
print('cache after:',stencil_offsets.cache_info())

radius 2: ((-2, 0), (-1, -1), (-1, 0), (-1, 1), (0, -2), (0, -1), (0, 0), (0, 1), (0, 2), (1, -1), (1, 0), (1, 1), (2, 0))
cache before: CacheInfo(hits=0, misses=1, maxsize=1024, currsize=1)
cache after: CacheInfo(hits=10, misses=1, maxsize=1024, currsize=1)


## 5. 与 JAX 的衔接：编译的计算函数不是 Python 字典遍历

JAX 的核心数值计算通常以同类型数组为主。你可以在 **CPU/Python 侧** 用字典描述静态配置信息，但不要指望把包含动态插入、删除的 Python 哈希表直接放进 `jit` 的设备计算内核。合理分工：

- CPU 侧：网格拓扑构建、元数据管理、任务调度；
- 数值侧：`ndarray` / JAX Array 中的密集或稀疏数值运算；
- 稀疏大型问题：后续学习专用稀疏矩阵与求解器。

---
## 6. 编程练习（先做再看答案）

练习 1：写 `unique_in_order(items)`，返回按**首次出现顺序**去重的列表。要求用 `set` 跟踪已见内容，元素为可哈希对象。

In [8]:
def unique_in_order(items):
    # TODO: 保存顺序与去重状态，保持输入不变
    return None

### 练习 2：构建二维网格的边界标签字典

写 `boundary_tags(ny, nx)`，返回 `{(j,i): 标签}`：左侧 `inlet`，右侧 `outlet`，其余最上和最下边 `wall`。**角点以左右侧优先**；要求 `ny,nx >= 2`。

In [9]:
def boundary_tags(ny, nx):
    # TODO: 角点的优先级很重要
    return None

### 练习 3：累计重复访问的网格单元

写 `count_cell_hits(path)`，输入一段由 `(j,i)` 元组构成的路径，返回访问次数 `dict`，不能改变原始列表。

In [10]:
def count_cell_hits(path):
    # TODO: 字典计数
    return None

## 练习自测

运行下方测试。`TODO` 未完成时出现“尚未通过”属于正常情况；逐项填入你的实现并再次运行。本讲的测试检查正确性与边界条件，性能优化可在通过后继续尝试。

In [11]:
def check(name, test_fn):
    try:
        test_fn()
        print(f"✅ {name}：通过")
    except Exception as exc:
        print(f"⬜ {name}：尚未通过 — {type(exc).__name__}: {exc}")

In [12]:
def test_ex1():
    assert unique_in_order([3,1,3,2,1,4]) == [3,1,2,4]
    assert unique_in_order([]) == []
    assert unique_in_order(['a','b','a']) == ['a','b']

def test_ex2():
    b = boundary_tags(4, 6)
    assert isinstance(b,dict)
    assert b[(0,0)] == 'inlet' and b[(3,0)] == 'inlet'
    assert b[(0,5)] == 'outlet' and b[(3,5)] == 'outlet'
    assert b[(0,2)] == 'wall' and b[(3,2)] == 'wall'
    assert (1,2) not in b and len(b) == 2*4+2*(6-2)
    assert boundary_tags(2,2) == {(0,0):'inlet',(1,0):'inlet',
                                    (0,1):'outlet',(1,1):'outlet'}

def test_ex3():
    p = [(1,1),(0,1),(1,1),(2,3),(1,1)]
    snap = p[:]
    assert count_cell_hits(p) == {(1,1):3,(0,1):1,(2,3):1}
    assert count_cell_hits([]) == {}
    assert p == snap

check('练习 1 / 按出现顺序去重',test_ex1)
check('练习 2 / CFD 边界标签',test_ex2)
check('练习 3 / 单元访问计数',test_ex3)

⬜ 练习 1 / 按出现顺序去重：尚未通过 — AssertionError: 
⬜ 练习 2 / CFD 边界标签：尚未通过 — AssertionError: 
⬜ 练习 3 / 单元访问计数：尚未通过 — AssertionError: 


## 7. 参考答案（独立命名，不覆盖你的代码）

先对照算法结构，再检查你对重复输入、角点和空列表的处理。

In [13]:
def unique_in_order_solution(items):
    result, seen = [], set()
    for x in items:
        if x not in seen:
            seen.add(x)
            result.append(x)
    return result

def boundary_tags_solution(ny, nx):
    if ny < 2 or nx < 2:
        raise ValueError('ny, nx must be >= 2')
    result = {}
    for j in range(ny):
        result[(j,0)] = 'inlet'
        result[(j,nx-1)] = 'outlet'
    for i in range(1,nx-1):
        result[(0,i)] = 'wall'
        result[(ny-1,i)] = 'wall'
    return result

def count_cell_hits_solution(path):
    hits = {}
    for cell in path:
        hits[cell] = hits.get(cell, 0) + 1
    return hits

assert unique_in_order_solution([2,2,1]) == [2,1]
assert boundary_tags_solution(2,2)[(1,1)] == 'outlet'
assert count_cell_hits_solution([(0,0),(0,0)]) == {(0,0):2}
print('✅ 参考答案自测通过')

✅ 参考答案自测通过


## 8. 小型综合实验：用哈希表建立稀疏网格编号

有时仅对活跃单元建立连续编号，便于后续组装线性方程组。这里使用 `dict[(j,i)] = k`，后续第 07 讲会把这些编号用于稀疏矩阵。

In [14]:
grid = [
    '.....',
    '.###.',
    '.....',
    '..#..'
]
active = [(j,i) for j,row in enumerate(grid) for i,ch in enumerate(row) if ch != '#']
cell_to_id = {cell: k for k, cell in enumerate(active)}
id_to_cell = {k: cell for cell,k in cell_to_id.items()}
assert all(id_to_cell[cell_to_id[c]] == c for c in active)
print('active count:', len(active),'sample mappings:',list(cell_to_id.items())[:8])

active count: 16 sample mappings: [((0, 0), 0), ((0, 1), 1), ((0, 2), 2), ((0, 3), 3), ((0, 4), 4), ((1, 0), 5), ((1, 4), 6), ((2, 0), 7)]


## 9. 自测与衔接

1. 为什么 `x in set` 的**平均**复杂度常常小于 `x in list`？
2. 为什么“可哈希”不等于“绝对不可变”？
3. 固定规则网格上的百万个浮点速度值为什么不适合逐项放进 `dict`？
4. 为什么用 `0.1+0.2` 这种浮点数作为关键几何坐标键容易遇到问题？
5. Python `dict` 和 JAX Array 在数值计算中适合承担什么不同角色？

**选做**：建立 `cell_to_id` 与 `id_to_cell` 映射；想想改变网格拓扑后，缓存为什么需要失效。下一讲学习容器内部的“指针连接”和动态扩容。